# Analyse en Voorspellingen van Wedstrijden in de Jupiler Pro League

**Academisch project / Data-analyse notebook**

Auteur: Torben de Schrijver
Datum: [2026]

Beschrijving: Deze notebook is bedoeld om factoren te analyseren die voetbalwedstrijden beïnvloeden, en om voorspellingen van experts te vergelijken met datagedreven modellen.

## 0. Imports


In [220]:
import pyodbc
import pandas as pd
from datetime import datetime

## 1. Introductie

### Doel van de analyse
Het doel van deze analyse is te onderzoeken hoe expertinschattingen van voetbalwedstrijden overeenkomen met data-gedreven voorspellingen op basis van historische wedstrijdstatistieken.

### Onderzoeksvraag
In welke mate zijn de voorspellingen van experts (journalisten, analisten, trainers) accuraat vergeleken met analyses gemaakt met programmeercode en machine learning modellen?

### Dataset beschrijving
De dataset bevat historische wedstrijddata van de Jupiler Pro League, inclusief:
- Wedstrijdresultaten
- Aantal doelpunten en kaarten
- Teamstatistieken (schoten, corners, ranglijstpositie, thuis/uit)
- Weersomstandigheden en stadioninformatie  
Daarnaast bevat de dataset ingevulde vragenlijsten van experts met hun voorspellingen en factorbeoordelingen.

### Overzicht van de methode
1. **Data verzamelen**: Historische wedstrijden en ingevulde vragenlijsten van experts.  
2. **Feature selectie**: Factoren zoals thuisvoordeel, recente vorm, rangverschil, weersomstandigheden.  
3. **Vergelijking voorspellingen**: Experts vs data-gedreven modellen.  
4. **Analyse**: Accuracy, bias-analyse, invloed van factoren en correlaties tussen expertinschattingen en resultaten.  
5. **Visualisatie en rapportage**: Grafieken, tabellen en samenvatting van inzichten.

## 2. Data laden

- Wedstrijddataset
- Weersdata
- Teams / Ranglijst
- Experts dataset

Commentaar over de structuur van de data

### Database connectie instellen

In [221]:
server = 'localhost'
database = 'JPL_Data'
username = 'tds'
password = 'tds'

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)
conn = pyodbc.connect(conn_str)

### Tabellen inladen

In [222]:
# Team
team_df = pd.read_sql("SELECT * FROM Team;", conn)

# Wedstrijd
wedstrijd_df = pd.read_sql("SELECT * FROM Wedstrijd;", conn)

# WedstrijdStatistiek
statistiek_df = pd.read_sql("SELECT * FROM WedstrijdStatistiek;", conn)

# WedstrijdWeer
weer_df = pd.read_sql("SELECT * FROM WedstrijdWeer;", conn)

# WedstrijdWeerVooraf
weer_vooraf_df = pd.read_sql("SELECT * FROM WedstrijdWeerVooraf;", conn)

C:\Users\TDS\AppData\Local\Temp\ipykernel_9112\1562388262.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  team_df = pd.read_sql("SELECT * FROM Team;", conn)
C:\Users\TDS\AppData\Local\Temp\ipykernel_9112\1562388262.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  wedstrijd_df = pd.read_sql("SELECT * FROM Wedstrijd;", conn)
C:\Users\TDS\AppData\Local\Temp\ipykernel_9112\1562388262.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  statistiek_df = pd.read_sql("SELECT * FROM WedstrijdStatistiek;", conn)
C:\Users\TDS\Ap

### CSV (enquête) inladen

In [223]:
enquete_df = pd.read_csv('../data/experts_enquete.csv')

## 3. Data cleaning

### Wedstrijddata

In [224]:
wedstrijd_df.info()

wedstrijd_df = wedstrijd_df.drop(['divisie'], axis=1)


<class 'pandas.DataFrame'>
RangeIndex: 1998 entries, 0 to 1997
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   wedstrijd_id     1998 non-null   int64 
 1   divisie          1998 non-null   str   
 2   wedstrijd_datum  1998 non-null   object
 3   wedstrijd_tijd   1998 non-null   object
 4   thuis_team       1998 non-null   str   
 5   uit_team         1998 non-null   str   
dtypes: int64(1), object(2), str(3)
memory usage: 93.8+ KB


In [225]:
# 1. Selecteer float64 kolommen
float_cols = statistiek_df.select_dtypes(include=['float64']).columns

# 2. Vul NaN met gemiddelde van elke kolom
statistiek_df[float_cols] = statistiek_df[float_cols].fillna(statistiek_df[float_cols].mean())

# 3. Rond af en converteer naar gewone int64
statistiek_df[float_cols] = statistiek_df[float_cols].round(0).astype('int')

# Check
statistiek_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1998 entries, 0 to 1997
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   wedstrijd_id               1998 non-null   int64
 1   doelpunten_thuis_voltijd   1998 non-null   int64
 2   doelpunten_uit_voltijd     1998 non-null   int64
 3   resultaat_voltijd          1998 non-null   str  
 4   doelpunten_thuis_halftijd  1998 non-null   int64
 5   doelpunten_uit_halftijd    1998 non-null   int64
 6   resultaat_halftijd         1998 non-null   str  
 7   schoten_thuis              1998 non-null   int64
 8   schoten_uit                1998 non-null   int64
 9   schoten_op_doel_thuis      1998 non-null   int64
 10  schoten_op_doel_uit        1998 non-null   int64
 11  overtredingen_thuis        1998 non-null   int64
 12  overtredingen_uit          1998 non-null   int64
 13  corners_thuis              1998 non-null   int64
 14  corners_uit                1998 non

In [226]:
team_df.info()

team_df = team_df.drop(['latitude', 'longitude'], axis=1)

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   team_naam           24 non-null     str    
 1   latitude            24 non-null     float64
 2   longitude           24 non-null     float64
 3   stadion_capaciteit  24 non-null     int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 896.0 bytes


In [227]:
weer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1998 entries, 0 to 1997
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   wedstrijd_id                    1998 non-null   int64  
 1   temperatuur_c                   1998 non-null   float64
 2   neerslag_mm                     1998 non-null   float64
 3   relatieve_luchtvochtigheid_pct  1998 non-null   int64  
 4   windsnelheid_m_s                1998 non-null   float64
 5   windrichting_graden             1998 non-null   int64  
 6   windstoten_m_s                  1998 non-null   float64
 7   bewolking_pct                   1998 non-null   int64  
 8   weercode                        1998 non-null   int64  
 9   luchtdruk_hpa                   1998 non-null   float64
 10  temperatuur_gem_c               1998 non-null   float64
 11  neerslag_som_mm                 1998 non-null   float64
 12  windsnelheid_gem_m_s            1998 non-null

In [228]:
weer_vooraf_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1998 entries, 0 to 1997
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   wedstrijd_id                  1998 non-null   int64  
 1   temperatuur_gem_laatste48_c   1998 non-null   float64
 2   neerslag_som_laatste48_mm     1998 non-null   float64
 3   windstoten_max_laatste48_m_s  1998 non-null   float64
 4   regen_uren_laatste48          1998 non-null   int64  
 5   hitte_uren_laatste48          1998 non-null   int64  
dtypes: float64(3), int64(3)
memory usage: 93.8 KB


#### Tabellen mergen

In [229]:
wedstrijden_compleet_df = (
    wedstrijd_df
    .merge(statistiek_df, on='wedstrijd_id', how='left')
    .merge(weer_df, on='wedstrijd_id', how='left')
    .merge(weer_vooraf_df, on='wedstrijd_id', how='left')
)

# merge met team info (thuis en uit)
wedstrijden_compleet_df = (
    wedstrijden_compleet_df
    .merge(team_df.add_prefix('thuis_'), left_on='thuis_team', right_on='thuis_team_naam', how='left')
    .merge(team_df.add_prefix('uit_'), left_on='uit_team', right_on='uit_team_naam', how='left')
)

wedstrijden_compleet_df = wedstrijden_compleet_df.drop('wedstrijd_id', axis=1)

print("Shape van complete dataset:", wedstrijden_compleet_df.shape)
wedstrijden_compleet_df.info()
wedstrijden_compleet_df.head()

Shape van complete dataset: (1998, 44)
<class 'pandas.DataFrame'>
RangeIndex: 1998 entries, 0 to 1997
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   wedstrijd_datum                 1998 non-null   object 
 1   wedstrijd_tijd                  1998 non-null   object 
 2   thuis_team                      1998 non-null   str    
 3   uit_team                        1998 non-null   str    
 4   doelpunten_thuis_voltijd        1998 non-null   int64  
 5   doelpunten_uit_voltijd          1998 non-null   int64  
 6   resultaat_voltijd               1998 non-null   str    
 7   doelpunten_thuis_halftijd       1998 non-null   int64  
 8   doelpunten_uit_halftijd         1998 non-null   int64  
 9   resultaat_halftijd              1998 non-null   str    
 10  schoten_thuis                   1998 non-null   int64  
 11  schoten_uit                     1998 non-null   int64  
 12  schote

,wedstrijd_datum,wedstrijd_tijd,thuis_team,uit_team,doelpunten_thuis_voltijd,doelpunten_uit_voltijd,resultaat_voltijd,doelpunten_thuis_halftijd,doelpunten_uit_halftijd,resultaat_halftijd,...,windstoten_max_m_s,temperatuur_gem_laatste48_c,neerslag_som_laatste48_mm,windstoten_max_laatste48_m_s,regen_uren_laatste48,hitte_uren_laatste48,thuis_team_naam,thuis_stadion_capaciteit,uit_team_naam,uit_stadion_capaciteit
0,2019-07-26,19:30:00,Genk,Kortrijk,2,1,H,0,1,A,...,35.6,30.6,0.0,37.4,0,28,Genk,23718,Kortrijk,9399
1,2019-07-27,17:00:00,Cercle Brugge,Standard,0,2,A,0,0,D,...,28.1,25.9,0.0,45.4,0,31,Cercle Brugge,29062,Standard,27670
2,2019-07-27,19:00:00,St Truiden,Mouscron,0,1,A,0,1,A,...,28.1,29.0,0.3,31.3,1,29,St Truiden,14600,Mouscron,12800
3,2019-07-27,19:00:00,Waregem,Mechelen,0,2,A,0,1,A,...,21.2,25.8,0.1,38.9,1,29,Waregem,12414,Mechelen,16672
4,2019-07-27,19:30:00,Waasland-Beveren,Club Brugge,1,3,A,1,1,D,...,24.5,26.8,0.0,38.5,0,28,Waasland-Beveren,8190,Club Brugge,29062


### Enquête

In [230]:
# 1-5 schaalvragen
scale_1_5 = [
    "Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?",
    "In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?  ",
    "In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?  ",
    "Heeft regen invloed op het aantal doelpunten?  ",
    "In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt? ",
    "Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat? ",
    "Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd? ",
    "In welke mate hebben eerdere confrontaties invloed in uw voorspelling? ",
    "In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd? ",
    "Hoe zeker bent u van deze match voorspelling?",
    "Hoeveel invloed heeft een derby/rivalen match op het aantal kaarten?",
    "Hoeveel invloed heeft een wedstrijd voor de titel op het aantal schoten?",
    "Hoeveel invloed heeft een wedstrijd voor degradatie op het aantal kaarten?"
]

# percentage vragen
scale_0_100 = [
    "Kans op thuiswinst  (0 - 100)",
    "Kans op gelijkspel  (0 - 100)",
    "Kans op uitwinst  (0 - 100)",
    "Als team A 10 plaatsen boven team B staat in het klassement: hoe groot is volgens u de kans op een overwinning van team A?  Percentage (0 - 100)",
    "Stel dat een team 3 keer op rij verloren heeft, hoe groot is volgens u de kans op een 4e nederlaag?  Percentage (0 - 100)"
]

# ===============================
# 2. Controle 1-5 schaal
# ===============================

for col in scale_1_5:
    if col in enquete_df.columns:
        enquete_df[col] = pd.to_numeric(enquete_df[col], errors="coerce")

        fouten = enquete_df[
            (enquete_df[col] < 1) |
            (enquete_df[col] > 5) |
            (enquete_df[col] % 1 != 0)
        ]

        if len(fouten) > 0:
            print(f"Foutieve waarden in kolom: {col}")
            print(fouten[[col]])

# ===============================
# 3. Controle 0-100 schaal
# ===============================

for col in scale_0_100:
    if col in enquete_df.columns:
        enquete_df[col] = pd.to_numeric(enquete_df[col], errors="coerce")

        fouten = enquete_df[
            (enquete_df[col] < 0) |
            (enquete_df[col] > 100) |
            (enquete_df[col] % 1 != 0)
        ]

        if len(fouten) > 0:
            print(f"Foutieve waarden in kolom: {col}")
            print(fouten[[col]])

enquete_df["kans_som_match1"] = (
    enquete_df["Kans op thuiswinst  (0 - 100)"] +
    enquete_df["Kans op gelijkspel  (0 - 100)"] +
    enquete_df["Kans op uitwinst  (0 - 100)"]
)

fouten = enquete_df[enquete_df["kans_som_match1"] != 100]
print(fouten)

Empty DataFrame
Columns: [Tijdstempel, Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?, In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?  , In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?  , Heeft regen invloed op het aantal doelpunten?  , In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt? , Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat? , Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd? , In welke mate hebben eerdere confrontaties invloed in uw voorspelling? , In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd? , Kans op thuiswinst  (0 - 100), Kans op gelijkspel  (0 - 100), Kans op uitwinst  (0 - 100), Verwacht aantal doelpunten thuisteam?, Verwacht aantal doelpunten uitteam?, Hoe zeker bent u van deze match voorspelling?, Kans op thuis

In [231]:
enquete_df.info()

enquete_df = enquete_df.drop(['Tijdstempel', 'kans_som_match1'], axis=1)

enquete_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 50 columns):
 #   Column                                                                                                                                            Non-Null Count  Dtype
---  ------                                                                                                                                            --------------  -----
 0   Tijdstempel                                                                                                                                       5 non-null      str  
 1   Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?                                                                        5 non-null      int64
 2   In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?                                                            5 non-null      int64
 3   In welke mate beïnvloedt stadioncapacite

,Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?,In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?,In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?,Heeft regen invloed op het aantal doelpunten?,In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt?,Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat?,Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd?,In welke mate hebben eerdere confrontaties invloed in uw voorspelling?,In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd?,Kans op thuiswinst (0 - 100),...,Rangschik de onderstaande factoren van meeste invloed → minste invloed voor aantal kaarten [5e],Rangschik de onderstaande factoren van meeste invloed → minste invloed voor aantal kaarten [6e],"Als het hard regent, zullen er ... kaarten verwacht worden.","Als het hard regent, zullen er ... doelpunten verwacht worden.",Als het thuisteam een aanvallende speelstijl hanteert en het uitteam defensief speelt zullen er gemiddeld ... doelpunten vallen.,"Als het erg warm (>23°C) is, zal het aantal schoten ... zijn dan gemiddeld.","Als er een derby is, of rivalen spelen, zal het aantal schoten van beide teams ... zijn dan gemiddeld.",Hoeveel invloed heeft een derby/rivalen match op het aantal kaarten?,Hoeveel invloed heeft een wedstrijd voor de titel op het aantal schoten?,Hoeveel invloed heeft een wedstrijd voor degradatie op het aantal kaarten?
0,4,1,1,3,2,4,4,4,5,50,...,stadioncapaciteit,tijdstip,meer,zelfde,meer,zelfde,meer,4,4,4
1,3,2,2,3,2,4,4,4,4,52,...,stadioncapaciteit,Recente vorm,meer,zelfde,zelfde,minder,meer,5,4,4
2,4,3,3,2,1,4,4,3,1,65,...,neerslag,tijdstip,zelfde,zelfde,meer,zelfde,minder,4,4,2
3,4,4,5,1,3,3,3,4,4,60,...,neerslag,stadioncapaciteit,meer,zelfde,minder,minder,meer,5,5,5
4,5,3,1,3,1,1,5,4,1,70,...,derby/rivalen,tijdstip,zelfde,zelfde,zelfde,zelfde,zelfde,4,1,1


## 4. Feature engineering

- Wedstrijdresultaat
- Rangverschil
- Huidige vorm
- Thuisvoordeel
- H2H resultaten
- Totale doelpunten
- Motivatievariabelen

Beschrijf welke features je maakt en waarom

## 5. Exploratory Data Analysis (EDA)

- Verdelen van thuis- en uitwinst
- Rangverschil versus resultaat
- Vormanalyse
- Weersomstandigheden
- Schoten / corners
- Grafieken en commentaar

## 6. Correlatieanalyse

- Factoren vs resultaten
- Factoren vs doelpunten
- Factoren vs kaarten
- Heatmaps of tabellen

## 7. Model voor wedstrijdresultaat

- Keuze model (logistische regressie, classificatie)
- Features en target
- Train-test split
- Resultaten evaluatie

## 8. Model voor doelpunten

- Keuze model (Poisson regression, regressie)
- Features en target
- Resultaten evaluatie

## 9. Model evaluatie

- Accuracy, log-loss, RMSE, MAE
- Grafieken van voorspelling vs echte resultaten

## 10. Expertdata importeren

- Inspectie expert dataset
- Beschrijving kolommen (probabilities, voorspelde doelpunten, confidence, motivatie)

## 11. Experts vs Model vergelijken

- Kansvoorspellingen (Brier score)
- Doelpunten (MAE / RMSE)
- Visualisatie (scatterplots, barplots)

## 12. Confidence vs Accuracy

- Confidence bins
- Accuracy per bin
- Calibration plots
- Analyse van over- of underconfidence

## 13. Bias analyse

- Thuisvoordeel overschatting
- Motivatie overschatting
- Vorm, rangverschil
- Histogrammen / visualisaties

## 14. Visualisaties

- Factor importance
- Experts vs model resultaten
- Doelpunten vergelijking
- Confidence calibration
- Interactieve grafieken (optioneel)

## 15. Conclusie / Discussie

- Belangrijkste bevindingen
- Factoren met grootste effect
- Verschillen experts vs model
- Limitaties
- Suggesties vervolgonderzoek